---
title: "In Class Assingment 1"
author: Karisa Kopecek
date: today
format:
  html:
    embed-resources: true
    echo: true
---

In [1]:
#importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn. linear_model import LinearRegression

In [2]:
# loading the dataset and initial EDA with SweetViz
insurance = pd.read_csv('insurance.csv')
insurance.head()

# visualizing the data with SweetViz
report = sv.analyze(insurance)
report.show_html('insurance_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [3]:
#encoding categorical variables
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [4]:
# preparing the data for modeling
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

# splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)

#scaling the features (don't need for RF but do for linear regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#baseline linear regression model
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

#baseline random forest model
baseline_rf = RandomForestRegressor(random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf) 

# print baseline results
print(f'Baseline Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}')
print(f'Baseline Random Forest R2: {mse_rf:.2f}, R2: {r2_rf:.2f}')

Baseline Linear Regression MSE: 39251253.82, R2: 0.74
Baseline Random Forest R2: 23527729.42, R2: 0.84


In [5]:
# CV with baseline models with r2 as the scoring metric
cv_scores_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring='r2')
cv_scores_rf = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring='r2')
print(f'Baseline Linear Regression CV R2 Scores: {cv_scores_lr}')
print(f'Baseline Random Forest CV R2 Scores: {cv_scores_rf}')   

Baseline Linear Regression CV R2 Scores: [0.75258897 0.76155328 0.7344671  0.748796   0.73283984]
Baseline Random Forest CV R2 Scores: [0.84314094 0.83348822 0.80238482 0.83554516 0.80344045]


In [6]:
# gridsearchCV for hyperparameter tuning of random forest
#chose to try to use my own values just to see difference between mine and yours
param_grid = {
    'n_estimators': [90, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [3, 5],
    'max_features': ['log2', 'sqrt']
}           

grid_search_rf = GridSearchCV(estimator=baseline_rf, 
                              param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)    
print(f'Best RF Hyperparameters: {grid_search_rf.best_params_}')
print(f'Best RF CV R2 Score: {grid_search_rf.best_score_:.2f}')

Best RF Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 3, 'n_estimators': 90}
Best RF CV R2 Score: 0.84


In [7]:
# examine top 20 ranked parameter combinations from grid search 
print("\nTop 20 GridSearchCV results:")
top20_eval = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20_eval)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
8             90        10                  3         log2         0.841977
11           200        10                  5         log2         0.841637
10            90        10                  5         log2         0.841002
3            200      None                  5         log2         0.840648
19           200        20                  5         log2         0.840648
9            200        10                  3         log2         0.839928
18            90        20                  5         log2         0.839570
2             90      None                  5         log2         0.839570
16            90        20                  3         log2         0.838075
0             90      None                  3         log2         0.838075
1            200      None                  3         log2         0.837964
17           200        20                  3         log2

I would choose the combination with **n_estimators = 90, max_depth = 10, min_samples_split = 3, and max_features = log2** because it achieves the highest mean cross-validation score (0.841977). However, several configurations have nearly identical performance, so the difference is small. As discussed in class if this were a real-world situation I would take a look at the data and decide if it is worth the computational cost and money to have a slight improvement. Since this data is just insurance data (not life-saving or something) I would probably still realistically pick n_estimators = 90, max_depth = 10, min_samples_split = 3, and max_features = log2, since it already uses fewer trees (making it faster to run) while still achieving the best performance among all tested models. 


In [8]:
# question 8
# tuned RF model using best parameters from GridSearchCV
tuned_rf = RandomForestRegressor(**grid_search_rf.best_params_, random_state=42)
tuned_rf.fit(X_train, y_train)

# predict on both training and test sets
y_pred_train_tuned = tuned_rf.predict(X_train)
y_pred_test_tuned = tuned_rf.predict(X_test)

# calculate R2 and RMSE for both
r2_train = r2_score(y_train, y_pred_train_tuned)
r2_test = r2_score(y_test, y_pred_test_tuned)
rmse_train = mean_squared_error(y_train, y_pred_train_tuned) ** 0.5
rmse_test = mean_squared_error(y_test, y_pred_test_tuned) ** 0.5

print("Tuned Random Forest -- Train vs Test Performance")
print(f"  Train R2:  {r2_train:.4f}  |  Test R2:  {r2_test:.4f}")
print(f"  Train RMSE: {rmse_train:.2f}  |  Test RMSE: {rmse_test:.2f}")
print(f"\n  R2 gap (train - test): {r2_train - r2_test:.4f}")
print(f"  RMSE gap (test - train): {rmse_test - rmse_train:.2f}")

Tuned Random Forest -- Train vs Test Performance
  Train R2:  0.9442  |  Test R2:  0.8586
  Train RMSE: 2849.36  |  Test RMSE: 4617.63

  R2 gap (train - test): 0.0856
  RMSE gap (test - train): 1768.27


The tuned Random Forest shows mild overfitting. The train R2 (0.9442) is higher
than the test R2 (0.8586) with a difference of 0.0856, and the RMSE jumps from 2,849 on training data
to 4,618 on test data (difference: 1,768). This means the model fits the training data
more closely than unseen data, which is the definition of overfitting. However, the overfitting is not severe. A test R2 of 0.86 still indicates strong predictive
performance. The gap would likely be larger with unconstrained trees, so I'm glad I used that. Overall,
the model generalizes pretty well but I could later maybe try increasing
min_samples_split or reducing n_estimators.